Điều kiện Mua (Entry LONG):

Giá nằm trên MA50 (Xác nhận phe Mua đang kiểm soát).

Giá có nhịp hồi (Dip) chạm lại hoặc nhúng nhẹ qua đường MA50.

Khớp lệnh khi có tín hiệu từ chối giảm (Ví dụ: Một cây nến rút chân/Pinbar hoặc nến xanh bao trùm ngay tại vùng MA50).

Dừng lỗ (Stop-Loss): Giá Entry - (1.5 x Giá trị ATR).

Chốt lời (Take-Profit): Giá Entry + (1.5 x Giá trị ATR)

In [ ]:
# Install required libraries
!pip install pandas pandas-ta requests plotly tabulate -q

import requests
import time
import pandas as pd
import pandas_ta as ta
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime

# ==========================================
# GLOBAL CONFIGURATION
# ==========================================
CONFIG = {
    'SYMBOLS': ['SOL-USD'],
    'INTERVAL': '1h',
    'START_DATE': '01/10/2020',
    'END_DATE': '01/05/2021',
    'INITIAL_CAPITAL': 10000.0,
    'POSITION_SIZE_PCT': 0.10,
    'FEE_RATE': 0.0004,
    'SLIPPAGE': 0.0002
}

# ==========================================
# MODULE 1: DATA HANDLER
# ==========================================
class BinanceDataFetcher:
    def __init__(self, symbol, interval, start_date, end_date):
        self.display_symbol = symbol.upper()
        self.api_symbol = self.display_symbol.replace('-USD', 'USDT')
        self.interval = interval
        self.start_date = start_date
        self.end_date = end_date

    def _str_to_millis(self, date_str: str) -> int:
        dt = datetime.strptime(date_str, "%d/%m/%Y")
        return int(dt.timestamp() * 1000)

    def fetch(self) -> pd.DataFrame:
        base_url = "https://data-api.binance.vision"

        print(f"Checking connection for {self.api_symbol}...")
        try:
            test = requests.get(f"{base_url}/api/v3/time", timeout=5)
            if test.status_code == 451:
                base_url = "https://api.binance.us"
        except Exception:
            base_url = "https://api.binance.us"

        url = base_url + "/api/v3/klines"
        start_time = self._str_to_millis(self.start_date)

        if self.end_date == datetime.now().strftime("%d/%m/%Y"):
            end_time = int(time.time() * 1000)
        else:
            end_dt = datetime.strptime(self.end_date, "%d/%m/%Y").replace(hour=23, minute=59, second=59)
            end_time = int(end_dt.timestamp() * 1000)

        all_klines = []
        print(f"Downloading {self.display_symbol} ({self.interval})...")

        while start_time < end_time:
            params = {"symbol": self.api_symbol, "interval": self.interval, "startTime": start_time, "endTime": end_time, "limit": 1000}
            try:
                r = requests.get(url, params=params, timeout=10)
                if r.status_code != 200: break
                data = r.json()
                if not data: break
                all_klines.extend(data)
                start_time = data[-1][0] + 1
                time.sleep(0.05)
            except Exception: break

        if not all_klines: return pd.DataFrame()

        df = pd.DataFrame(all_klines, columns=["timestamp", "Open", "High", "Low", "Close", "Volume", "CT", "QAV", "NT", "TBBV", "TBQV", "I"])
        df = df[["timestamp", "Open", "High", "Low", "Close", "Volume"]]
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
        df.set_index("timestamp", inplace=True)
        df = df.astype(float)
        return df

# ==========================================
# MODULE 2: STRATEGY LOGIC
# ==========================================
class MA50PullbackStrategy:
    def __init__(self):
        self.ma_period = 50
        self.atr_period = 14
        self.atr_multiplier = 1.5

    def calculate_indicators(self, df: pd.DataFrame) -> pd.DataFrame:
        df['MA_50'] = ta.sma(df['Close'], length=self.ma_period)
        df['ATR'] = ta.atr(df['High'], df['Low'], df['Close'], length=self.atr_period)
        df.dropna(inplace=True)
        return df

# ==========================================
# MODULE 3: BACKTEST ENGINE
# ==========================================
class BacktestEngine:
    def __init__(self, df, strategy, config):
        self.df = strategy.calculate_indicators(df.copy())
        self.strategy = strategy
        self.config = config

        self.capital = config['INITIAL_CAPITAL']
        self.invested_amount = 0.0
        self.position = 0
        self.entry_price = 0.0
        self.sl_price = 0.0
        self.tp_price = 0.0
        self.trades = []

        self.equity_curve = np.zeros(len(self.df))

    def run(self):
        O = self.df['Open'].values
        H = self.df['High'].values
        L = self.df['Low'].values
        C = self.df['Close'].values
        times = self.df.index

        ma50 = self.df['MA_50'].values
        atr = self.df['ATR'].values

        fee = self.config['FEE_RATE']
        slip = self.config['SLIPPAGE']
        pos_size_pct = self.config['POSITION_SIZE_PCT']

        s = self.strategy
        self.equity_curve[0] = self.capital
        self.equity_curve[1] = self.capital

        # Bắt đầu vòng lặp từ nến thứ 2 (i=2) để có thể check dữ liệu của 2 nến trước đó (Phục vụ mẫu hình Engulfing)
        for i in range(2, len(self.df)):
            # ---------------------------
            # EXIT LOGIC (Bracket Orders: TP & SL)
            # ---------------------------
            if self.position == 1:
                exit_signal = False
                reason = ""
                real_exit = 0.0

                if L[i] <= self.sl_price:
                    exit_signal = True
                    reason = "SL (-1.5x ATR)"
                    real_exit = min(O[i], self.sl_price) * (1 - slip)

                elif H[i] >= self.tp_price:
                    exit_signal = True
                    reason = "TP (+1.5x ATR)"
                    real_exit = max(O[i], self.tp_price) * (1 - slip)

                if exit_signal:
                    pnl_pct = (real_exit - self.entry_price) / self.entry_price
                    net_pnl_pct = pnl_pct - (fee * 2)
                    pnl_amt = self.invested_amount * net_pnl_pct
                    self.capital += pnl_amt

                    self.trades.append({
                        'Date': times[i], 'Type': 'Long', 'Reason': reason,
                        'Entry': self.entry_price, 'Exit': real_exit,
                        'PnL_Amt': pnl_amt, 'Equity': self.capital
                    })

                    self.position = 0
                    self.invested_amount = 0.0
                    self.sl_price = 0.0
                    self.tp_price = 0.0

            # ---------------------------
            # ENTRY LOGIC (PA Pullback MA50)
            # ---------------------------
            if self.position == 0:
                # 1. Điều kiện 1: Có nhúng chạm/phá giả MA50 và Đóng nến thành công trên MA50 (Phe mua đỡ giá)
                touched_ma = (L[i-1] <= ma50[i-1]) or (L[i-2] <= ma50[i-2])
                bulls_in_control = C[i-1] > ma50[i-1]

                # 2. Điều kiện 2: Nhận diện mô hình nến đảo chiều (Price Action)
                # Mô hình A: Bullish Pinbar (Nến búa xanh) tại nến liền trước
                body_1 = abs(C[i-1] - O[i-1])
                lower_wick_1 = min(C[i-1], O[i-1]) - L[i-1]
                is_pinbar = (C[i-1] > O[i-1]) and (lower_wick_1 >= 1.5 * body_1)

                # Mô hình B: Bullish Engulfing (Nến bao trùm tăng)
                prev_red = C[i-2] < O[i-2]
                curr_green = C[i-1] > O[i-1]
                is_engulfing = prev_red and curr_green and (C[i-1] >= O[i-2]) and (O[i-1] <= C[i-2])

                # KÍCH HOẠT LỆNH
                if touched_ma and bulls_in_control and (is_pinbar or is_engulfing):
                    self.position = 1
                    self.entry_price = O[i] * (1 + slip)

                    # Set TP/SL 1:1 bằng 1.5 * ATR
                    atr_dist = s.atr_multiplier * atr[i-1]
                    self.sl_price = self.entry_price - atr_dist
                    self.tp_price = self.entry_price + atr_dist

                    self.invested_amount = self.capital * pos_size_pct

            # TÍNH VỐN UNREALIZED
            current_equity = self.capital
            if self.position == 1:
                current_value = self.invested_amount * (C[i] / self.entry_price)
                unrealized_pnl = current_value - self.invested_amount
                current_equity += unrealized_pnl

            self.equity_curve[i] = current_equity

        return pd.Series(self.equity_curve, index=self.df.index), pd.DataFrame(self.trades)

# ==========================================
# MODULE 4: PERFORMANCE EVALUATOR
# ==========================================
class Evaluator:
    @staticmethod
    def evaluate(equity, trades_df, initial_capital, symbol, interval):
        if trades_df.empty:
            return None

        final_equity = equity.iloc[-1]
        net_pnl = final_equity - initial_capital
        total_return = net_pnl / initial_capital

        rolling_max = equity.cummax()
        drawdown = (equity - rolling_max) / rolling_max
        max_dd = drawdown.min()

        wins = trades_df[trades_df['PnL_Amt'] > 0]
        losses = trades_df[trades_df['PnL_Amt'] <= 0]
        winrate = len(wins) / len(trades_df) if len(trades_df) > 0 else 0
        profit_factor = wins['PnL_Amt'].sum() / abs(losses['PnL_Amt'].sum()) if len(losses) > 0 and losses['PnL_Amt'].sum() != 0 else 0

        def get_annual_factor(tf):
            if tf == '15m': return 365 * 24 * 4
            elif tf == '1h': return 365 * 24
            elif tf == '4h': return 365 * 6
            elif tf == '1d': return 365
            return 365 * 24

        annual_factor = get_annual_factor(interval)
        returns = equity.pct_change().dropna()
        sharpe_ratio = (returns.mean() / returns.std()) * np.sqrt(annual_factor) if returns.std() != 0 else 0

        return {
            'Symbol': symbol,
            'Total Trades': len(trades_df),
            'Winrate': f"{winrate:.2%}",
            'Profit Factor': f"{profit_factor:.2f}",
            'Max Drawdown': f"{max_dd:.2%}",
            'Sharpe Ratio': round(sharpe_ratio, 2),
            'Net PnL ($)': round(net_pnl, 2),
            'Total Return': f"{total_return:.2%}"
        }

    @staticmethod
    def plot_chart(equity, drawdown, symbol, interval):
        fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08, row_heights=[0.7, 0.3])
        fig.add_trace(go.Scatter(x=equity.index, y=equity.values, mode="lines", name="Equity", line=dict(color="#2962ff", width=1.5)), row=1, col=1)
        fig.add_trace(go.Scatter(x=drawdown.index, y=drawdown.values, mode="lines", name="Drawdown", line=dict(color="#d32f2f", width=1), fill='tozeroy', fillcolor='rgba(211, 47, 47, 0.2)'), row=2, col=1)

        fig.update_layout(title=f"Performance: {symbol} ({interval}) - MA50 Bounce PA", template="plotly_white", height=500, margin=dict(l=40, r=40, t=60, b=40), showlegend=False)
        fig.update_xaxes(showgrid=True, gridcolor='#e6e6e6', zeroline=False)
        fig.update_yaxes(title_text="Balance ($)", tickprefix="$", row=1, col=1)
        fig.update_yaxes(title_text="Drawdown %", tickformat=".1%", row=2, col=1)
        fig.show()

# ==========================================
# MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    summary_results = []

    for symbol in CONFIG['SYMBOLS']:
        print(f"\n[{symbol}] Processing...")
        fetcher = BinanceDataFetcher(symbol, CONFIG['INTERVAL'], CONFIG['START_DATE'], CONFIG['END_DATE'])
        df_raw = fetcher.fetch()

        if not df_raw.empty:
            my_strategy = MA50PullbackStrategy()
            engine = BacktestEngine(df_raw, my_strategy, CONFIG)
            equity, trades = engine.run()

            # Evaluate Metrics
            metrics = Evaluator.evaluate(equity, trades, CONFIG['INITIAL_CAPITAL'], symbol, CONFIG['INTERVAL'])

            if metrics:
                summary_results.append(metrics)
                rolling_max = equity.cummax()
                drawdown = (equity - rolling_max) / rolling_max
                Evaluator.plot_chart(equity, drawdown, symbol, CONFIG['INTERVAL'])
            else:
                print(f"[{symbol}] No trades executed.")

    # IN RA BẢNG TỔNG HỢP (SUMMARY TABLE)
    if summary_results:
        print("\n" + "="*80)
        print(" " * 25 + f"BACKTEST SUMMARY ({CONFIG['INTERVAL']})")
        print("="*80)
        summary_df = pd.DataFrame(summary_results)
        print(summary_df.to_markdown(index=False))
        print("="*80 + "\n")


[SOL-USD] Processing...
Checking connection for SOLUSDT...



                         BACKTEST SUMMARY (1h)
| Symbol   |   Total Trades | Winrate   |   Profit Factor | Max Drawdown   |   Sharpe Ratio |   Net PnL ($) | Total Return   |
|:---------|---------------:|:----------|----------------:|:---------------|---------------:|--------------:|:---------------|
| SOL-USD  |             89 | 55.06%    |             1.4 | -2.90%         |           1.68 |        679.78 | 6.80%          |



# Go Code

In [ ]:
%%writefile MAPullbackPAStrategy.go
package strategy

import "math"

// Candle represents one OHLCV data point
// ĐÃ THÊM: Open, High, Low để phục vụ Price Action và ATR
type Candle struct {
	Open  float64
	High  float64
	Low   float64
	Close float64
}

// Signal represents output signals for each index
type Signal struct {
	LongEntry  bool
	LongExit   bool
	ShortEntry bool
	ShortExit  bool
}

// MAPullbackPAStrategy struct
type MAPullbackPAStrategy struct {
	MAPeriod      int
	ATRPeriod     int
	ATRMultiplier float64
}

// Constructor
func NewMAPullbackPAStrategy(ma, atr int, mult float64) *MAPullbackPAStrategy {
	return &MAPullbackPAStrategy{
		MAPeriod:      ma,
		ATRPeriod:     atr,
		ATRMultiplier: mult,
	}
}

// --- SMA ---
func calculateSMA(data []float64, period int) []float64 {
	sma := make([]float64, len(data))
	var sum float64

	for i := 0; i < len(data); i++ {
		sum += data[i]

		if i >= period {
			sum -= data[i-period]
		}

		if i >= period-1 {
			sma[i] = sum / float64(period)
		} else {
			sma[i] = math.NaN()
		}
	}

	return sma
}

// --- ATR (Sử dụng Wilder's Smoothing/RMA chuẩn như Pandas-TA) ---
func calculateATR(high, low, close []float64, period int) []float64 {
	n := len(close)
	atr := make([]float64, n)
	if n == 0 {
		return atr
	}

	// 1. Tính True Range (TR)
	tr := make([]float64, n)
	tr[0] = high[0] - low[0]
	for i := 1; i < n; i++ {
		hl := high[i] - low[i]
		hc := math.Abs(high[i] - close[i-1])
		lc := math.Abs(low[i] - close[i-1])
		tr[i] = math.Max(hl, math.Max(hc, lc))
	}

	// 2. Làm mượt TR bằng RMA (Wilder's Moving Average)
	sum := 0.0
	for i := 0; i < period && i < n; i++ {
		sum += tr[i]
	}

	if n >= period {
		atr[period-1] = sum / float64(period)
		alpha := 1.0 / float64(period)
		for i := period; i < n; i++ {
			atr[i] = alpha*tr[i] + (1-alpha)*atr[i-1]
		}
	}

	// Gán NaN cho các giá trị ban đầu chưa đủ chu kỳ
	for i := 0; i < period-1 && i < n; i++ {
		atr[i] = math.NaN()
	}

	return atr
}

// --- Apply Strategy ---
func (s *MAPullbackPAStrategy) Apply(candles []Candle) []Signal {
	n := len(candles)
	signals := make([]Signal, n)

	if n < s.MAPeriod || n < 3 {
		return signals // Không đủ nến để chạy
	}

	// Trích xuất dữ liệu mảng
	open := make([]float64, n)
	high := make([]float64, n)
	low := make([]float64, n)
	close := make([]float64, n)

	for i, c := range candles {
		open[i] = c.Open
		high[i] = c.High
		low[i] = c.Low
		close[i] = c.Close
	}

	ma50 := calculateSMA(close, s.MAPeriod)
	atr := calculateATR(high, low, close, s.ATRPeriod)

	// Biến theo dõi trạng thái lệnh (State Management)
	inPosition := false
	var slPrice, tpPrice float64

	// Bắt đầu từ 2 vì mẫu hình Engulfing cần kiểm tra (i-2) và (i-1)
	for i := 2; i < n; i++ {

		// 1. KIỂM TRA EXIT TRƯỚC (Stop Loss / Take Profit)
		if inPosition {
			hitSL := low[i] <= slPrice
			hitTP := high[i] >= tpPrice

			if hitSL || hitTP {
				signals[i].LongExit = true
				inPosition = false
				slPrice = 0
				tpPrice = 0
				continue // Đã thoát lệnh thì nhảy sang nến tiếp theo, không Entry lại ở cùng 1 nến
			}
		}

		// 2. KIỂM TRA ENTRY (Chỉ check khi đang không có lệnh)
		if !inPosition {
			// Bỏ qua nếu chỉ báo chưa kịp tính ra số (NaN)
			if math.IsNaN(ma50[i-1]) || math.IsNaN(atr[i-1]) {
				continue
			}

			// Điều kiện 1: MA Context (Giá đè chạm MA50 nhưng đóng trên MA50)
			touchedMA := (low[i-1] <= ma50[i-1]) || (low[i-2] <= ma50[i-2])
			bullsControl := close[i-1] > ma50[i-1]

			// Điều kiện 2: Price Action
			// A. Nhận diện Pinbar (Rút chân xanh)
			body1 := math.Abs(close[i-1] - open[i-1])
			lowerWick1 := math.Min(close[i-1], open[i-1]) - low[i-1]
			isPinbar := (close[i-1] > open[i-1]) && (lowerWick1 >= 1.5*body1)

			// B. Nhận diện Engulfing (Nuốt chửng tăng)
			prevRed := close[i-2] < open[i-2]
			currGreen := close[i-1] > open[i-1]
			isEngulfing := prevRed && currGreen && (close[i-1] >= open[i-2]) && (open[i-1] <= close[i-2])

			// KÍCH HOẠT LỆNH LONG
			if touchedMA && bullsControl && (isPinbar || isEngulfing) {
				signals[i].LongEntry = true
				inPosition = true

				// Tính toán mốc TP và SL tĩnh tại thời điểm Entry (Mở nến hiện tại)
				entryPrice := open[i]
				atrDist := s.ATRMultiplier * atr[i-1]

				slPrice = entryPrice - atrDist
				tpPrice = entryPrice + atrDist
			}
		}
	}

	return signals
}